# 05 — Generate 1,000,000 incidents (Spark)

Builds `synth_incidents` with the columns of `maintops.incidents`, fully in Spark (no Python loops), as a Delta table with Change Data Feed on.

How the rows are made consistent:
* **Client**: random client (`is_handyman = false`, ids `1..N_USERS`), skewed so some clients report far more incidents than others. Incident text comes from per-type problem templates.
* **Type / urgency**: type follows `SPEC_WEIGHTS`; urgency comes from the chosen problem (a burst pipe is `critical`, a dripping tap `low`) with some random shift.
* **Handyman**: picked from handymen in the **client's region** who have the incident type as a specialisation. Higher-`quality` handymen are picked more often.
* **Distance / travel time**: haversine between client and handyman × 1.3 road factor, at 22–38 km/h city speed.
* **Status mix**: ~92 % completed, ~6 % cancelled, ~2 % still active (open / recommended / assigned / in_progress, created in the last days).
* **Rating / feedback**: ~88 % of completed incidents are rated; rating follows the handyman's hidden quality plus noise.
* Timestamps (`created_at`, `assigned_at`, `completed_at`, `updated_at`) follow the order created → assigned → completed; every account is older than the first incident.
* Handyman statistics (`completed_cases`, `rating_avg`, `rating_count`) are computed by the Spark pipeline (`pipeline/13_gold_performance` → Lakebase), not here.

In [1]:
# Environment bootstrap: runs unchanged on Databricks and on a local machine.
# Locally it connects through Databricks Connect (serverless unless DATABRICKS_CLUSTER_ID is set)
# and provides `spark`, `display` and `list_volume_files`. Override the profile with DATABRICKS_CONFIG_PROFILE.
import os
IS_DATABRICKS = "DATABRICKS_RUNTIME_VERSION" in os.environ
LOCAL_SPARK_DEPENDENCIES = []   # pip packages the remote workers need (serverless Databricks Connect only)
if not IS_DATABRICKS:
    os.environ.setdefault("DATABRICKS_CONFIG_PROFILE", "george.sokolovsky2001@gmail.com")
    from databricks.connect import DatabricksSession

    _builder = DatabricksSession.builder
    if os.environ.get("DATABRICKS_CLUSTER_ID"):
        _builder = _builder.clusterId(os.environ["DATABRICKS_CLUSTER_ID"])
    else:
        _builder = _builder.serverless()
        if LOCAL_SPARK_DEPENDENCIES:
            from databricks.connect import DatabricksEnv
            _builder = _builder.withEnvironment(DatabricksEnv().withDependencies(LOCAL_SPARK_DEPENDENCIES))
    spark = _builder.getOrCreate()

    def display(obj, n=1000):
        from IPython.display import display as _display
        _display(obj.limit(n).toPandas() if hasattr(obj, "toPandas") else obj)

def list_volume_files(directory, suffix=""):
    """File names in a Unity Catalog Volume folder (the Volume is only mounted on Databricks, so go through the Files API locally)."""
    if IS_DATABRICKS:
        return [f for f in os.listdir(directory) if f.endswith(suffix)] if os.path.isdir(directory) else []
    from databricks.sdk import WorkspaceClient
    return [e.name for e in WorkspaceClient().files.list_directory_contents(directory) if e.name.endswith(suffix)]


In [2]:
%run ./00_config

In [3]:
# Cell 1 — Helpers for random column expressions (each call to rnd() gets its own seed, so runs are reproducible)
import math
from pyspark.sql import functions as F

_seed = [SEED + 100]
def rnd():
    """A fresh, independently seeded rand() column."""
    _seed[0] += 1
    return F.rand(_seed[0])

def pick(values, u=None):
    """Random element of a python list, as a column."""
    u = rnd() if u is None else u
    return F.element_at(F.array(*[F.lit(v) for v in values]), (F.floor(u * len(values)) + 1).cast("int"))

def weighted(weights):
    """Categorical draw from {value: probability}."""
    u, cum, expr = rnd(), 0.0, None
    items = list(weights.items())
    for k, w in items[:-1]:
        cum += w
        expr = F.when(u < cum, F.lit(k)) if expr is None else expr.when(u < cum, F.lit(k))
    return expr.otherwise(F.lit(items[-1][0]))

def epoch(ts): return F.unix_timestamp(F.lit(ts))
def to_ts(sec_col): return sec_col.cast("long").cast("timestamp")

STATUS_WEIGHTS = {"completed": .92, "cancelled": .06, "in_progress": .005, "assigned": .005, "recommended": .005, "open": .005}
ACTIVE = ["open", "recommended", "assigned", "in_progress"]

In [4]:
# Cell 2 — Base rows: client, type, status
# user_id = floor(N * u**1.3) + 1 favours low ids, so a few clients report many more incidents than the rest
users = spark.table(T_USERS).where("NOT is_handyman").select(F.col("id").alias("user_id"), F.col("state").alias("u_state"),
                                    F.col("latitude").alias("u_lat"), F.col("longitude").alias("u_lon"))

inc = (spark.range(1, N_INCIDENTS + 1)
       .withColumn("user_id", (F.floor(F.lit(N_USERS) * F.pow(rnd(), 1.3)) + 1).cast("long"))
       .withColumn("incident_type", weighted(SPEC_WEIGHTS))
       .withColumn("status", weighted(STATUS_WEIGHTS))
       .join(F.broadcast(users), "user_id"))

In [5]:
# Cell 3 — Candidate pools: handymen per (client region, specialisation), best quality first
#
# Each pool is an array of handyman ids sorted by hidden quality (best first), so an incident's handyman
# can later be drawn from it with a bias towards the front (see pool_pick).
#   pool_local  = handymen in the client's state with the incident type as a specialisation
#   pool_global = same skill, any state; used only when the client's state has nobody with that skill
truth = spark.table(T_HANDYMEN_TRUTH).select("user_id", "quality")

def make_pools(keys):
    # one row per (handyman, skill) -> group by `keys` -> ids sorted by quality descending
    return (spark.table(T_HANDYMAN_DETAILS).select("user_id", F.explode("specialisations").alias("spec"))
            .join(spark.table(T_USERS).where("is_handyman AND is_active").select(F.col("id").alias("user_id"), "state"), "user_id")
            .join(truth, "user_id").withColumnRenamed("user_id", "id")
            .groupBy(*keys).agg(F.sort_array(F.collect_list(F.struct("quality", "id")), asc=False).alias("s"))
            .withColumn("pool", F.transform("s", lambda x: x["id"])).drop("s"))

pool_local  = make_pools(["state", "spec"]).select(F.col("state").alias("u_state"), F.col("spec").alias("incident_type"), F.col("pool").alias("pool_l"))
pool_global = make_pools(["spec"]).select(F.col("spec").alias("incident_type"), F.col("pool").alias("pool_g"))

# pools are tiny (states x skills), so broadcast them; local pool wins, global is the fallback
inc = (inc.join(F.broadcast(pool_local), ["u_state", "incident_type"], "left")
          .join(F.broadcast(pool_global), "incident_type", "left")
          .withColumn("pool", F.coalesce("pool_l", "pool_g")).drop("pool_l", "pool_g"))

def pool_pick(skew):
    """Element of `pool`; skew > 1 favours the front (= the best-rated handymen).
    Index = floor(size * u**skew): skew 1 is uniform, higher skew pulls the index towards 0."""
    return F.element_at("pool", (F.floor(F.size("pool") * F.pow(rnd(), skew)) + 1).cast("int"))

In [6]:
# Cell 4 — Problem text, urgency, handyman and recommendations
# Each type has 10 problem templates, each with a base urgency (0-3) that is nudged by -1 / +1 at random.
problem_idx = (F.floor(rnd() * 10)).cast("int")
inc = inc.withColumn("problem_idx", problem_idx)

problem_text, base_urgency = None, None
for t, items in PROBLEMS.items():
    texts   = F.array(*[F.lit(p) for p, _ in items])
    urgency = F.array(*[F.lit(u) for _, u in items])
    cond = F.col("incident_type") == t
    problem_text = F.when(cond, F.element_at(texts, F.col("problem_idx") + 1)) if problem_text is None else problem_text.when(cond, F.element_at(texts, F.col("problem_idx") + 1))
    base_urgency = F.when(cond, F.element_at(urgency, F.col("problem_idx") + 1)) if base_urgency is None else base_urgency.when(cond, F.element_at(urgency, F.col("problem_idx") + 1))

# the two rnd() calls are independent, so roughly 12 % shift down and ~21 % shift up
shift   = F.when(rnd() < .12, -1).when(rnd() < .24, 1).otherwise(0)
lvl     = F.least(F.lit(3), F.greatest(F.lit(0), base_urgency + shift))
u_names = F.array(*[F.lit(x) for x in URGENCY_LEVELS])

inc = (inc.withColumn("problem", problem_text)
          .withColumn("urgency", F.element_at(u_names, lvl + 1))
          # handyman: only for statuses that have one (cancelled ones sometimes had one already)
          .withColumn("handyman_id", F.when(F.col("status").isin("assigned", "in_progress", "completed") | ((F.col("status") == "cancelled") & (rnd() < .35)), pool_pick(1.6)))
          # top-3 shortlist: always includes the chosen handyman (2 uniform picks fill the rest; duplicates collapse, so it can be < 3);
          # open incidents have none yet
          .withColumn("recommended_handyman_ids",
                      F.when(F.col("status") != "open",
                             F.shuffle(F.array_distinct(F.array(F.coalesce("handyman_id", pool_pick(1.6)), pool_pick(1.0), pool_pick(1.0))))))
          .drop("pool", "problem_idx"))

In [7]:
# Cell 5 — Description, reasoning, distance / travel time
# description = opener + problem + detail + urgency-specific tail; distance is haversine x 1.3 road factor
tails = None
for lvl_name, options in URGENCY_TAILS.items():
    cond = F.col("urgency") == lvl_name
    tails = F.when(cond, pick(options)) if tails is None else tails.when(cond, pick(options))

inc = (inc.withColumn("description", F.trim(F.concat_ws(" ",
                        F.concat(pick(OPENERS), F.col("problem"), F.lit(".")), pick(DETAILS), tails)))
          .withColumn("agent_reasoning",
                      F.when(F.col("status") != "open",
                             F.concat(F.lit("Classified as "), F.col("incident_type"), F.lit(" with "), F.col("urgency"),
                                      F.lit(" urgency from the client's description. Shortlisted "), F.size("recommended_handyman_ids"),
                                      F.lit(" handymen with a matching specialisation in "), F.col("u_state"),
                                      F.lit(", ranked by skill match, past feedback, current workload and travel time."))))
          .drop("problem"))

handy = spark.table(T_USERS).where("is_handyman").select(F.col("id").alias("handyman_id"), F.col("latitude").alias("h_lat"), F.col("longitude").alias("h_lon")) \
             .join(truth.withColumnRenamed("user_id", "handyman_id"), "handyman_id")
inc = inc.join(F.broadcast(handy), "handyman_id", "left")

rad = math.pi / 180
a = (F.pow(F.sin((F.col("h_lat") - F.col("u_lat")) * rad / 2), 2)
     + F.cos(F.col("u_lat") * rad) * F.cos(F.col("h_lat") * rad) * F.pow(F.sin((F.col("h_lon") - F.col("u_lon")) * rad / 2), 2))
distance = 2 * 6371 * F.asin(F.sqrt(a)) * 1.3
inc = (inc.withColumn("distance_km", F.round(distance, 2))
          .withColumn("travel_time_minutes", F.round(distance / (22 + rnd() * 16) * 60 + 2, 2)))

In [8]:
# Cell 6 — Timestamps, rating, feedback
# Timeline: created -> assigned (faster when urgent) -> completed (rnd()*rnd() skews towards quick jobs).
# Rating = 2.6 + 2.2 * hidden quality + noise, clamped to 1-5, so better handymen get better ratings.
is_active = F.col("status").isin(*ACTIVE)
created = F.when(is_active, epoch(ACTIVE_START) + rnd() * (epoch(ACTIVE_END) - epoch(ACTIVE_START))) \
           .otherwise(epoch(HISTORY_START) + rnd() * (epoch(HISTORY_END) - epoch(HISTORY_START)))
urgency_speed = F.when(F.col("urgency") == "critical", .1).when(F.col("urgency") == "high", .3).otherwise(1.0)

inc = (inc.withColumn("created_s", created)
          .withColumn("assigned_s", F.when(F.col("handyman_id").isNotNull(), F.col("created_s") + 120 + rnd() * 21600 * urgency_speed))
          .withColumn("completed_s", F.when(F.col("status") == "completed", F.col("assigned_s") + 3600 + rnd() * rnd() * 432000))
          .withColumn("updated_s", F.when(F.col("status") == "completed", F.col("completed_s"))
                                    .otherwise(F.coalesce("assigned_s", "created_s") + rnd() * F.when(F.col("status") == "cancelled", 86400).otherwise(1800)))
          .withColumn("created_at",   to_ts(F.col("created_s")))
          .withColumn("assigned_at",  to_ts(F.col("assigned_s")))
          .withColumn("completed_at", to_ts(F.col("completed_s")))
          .withColumn("updated_at",   to_ts(F.col("updated_s"))))

rated  = (F.col("status") == "completed") & (rnd() < .88)
rating = F.least(F.lit(5), F.greatest(F.lit(1), F.round(2.6 + 2.2 * F.col("quality") + F.randn(_seed[0] + 1) * 0.8))).cast("int")
inc = inc.withColumn("rating", F.when(rated, rating))

feedback = None
for stars, texts in FEEDBACK.items():
    cond = F.col("rating") == stars
    expr = F.concat(pick(texts), pick(FEEDBACK_EXTRAS))
    feedback = F.when(cond, expr) if feedback is None else feedback.when(cond, expr)
inc = inc.withColumn("feedback", feedback)

In [9]:
# Cell 7 — Final columns in the Lakebase order, write Delta
final = inc.select(
    F.col("id").cast("bigint").alias("id"),
    F.col("user_id").cast("bigint").alias("reported_by_user_id"),
    F.col("handyman_id").cast("bigint").alias("handyman_user_id"),
    "description",
    F.col("incident_type"),
    "urgency",
    F.col("recommended_handyman_ids").cast("array<bigint>").alias("recommended_handyman_ids"),
    "agent_reasoning",
    F.col("distance_km").cast("decimal(10,2)").alias("distance_km"),
    F.col("travel_time_minutes").cast("decimal(10,2)").alias("travel_time_minutes"),
    "status",
    "rating",
    "feedback",
    "created_at", "assigned_at", "completed_at", "updated_at",
)

(final.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(T_INCIDENTS))
spark.sql(f"ALTER TABLE {T_INCIDENTS} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
print(f"Incidents written: {spark.table(T_INCIDENTS).count():,}")

Incidents written: 1,000,000
